# Deep Dive: Agentic Retrieval Augmented Generation

Extended from the [Agentic Router notebook](https://github.com/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Production_Agentic_RAG_AI_Systems/001.%20Agentic%20Router.ipynb).

**Extensions implemented:**
- **Part 1 (required):** Sub-query division — compound questions split into independent sub-queries, each routed separately, results combined with renumbered citations.
- **Bonus:** RBAC-aware semantic caching — FAISS-backed cache partitioned by role, preventing information leakage across access levels.

## Setup and Dependencies

In [ ]:
!pip install -q openai qdrant-client transformers==4.48.0 tavily-python faiss-cpu torch

In [ ]:
import os
import re
import json
import asyncio
import numpy as np

import faiss
import nest_asyncio
from openai import OpenAI, OpenAIError
from transformers import AutoTokenizer, AutoModel
import qdrant_client
from tavily import TavilyClient

nest_asyncio.apply()

## API Keys

Set these environment variables before running, or replace with your keys directly (not recommended for shared notebooks):
```bash
export OPENAI_API_KEY="sk-..."
export TAVILY_API_KEY="tvly-..."
```

In [ ]:
openai_api_key = os.environ.get("OPENAI_API_KEY", "")
tavily_api_key = os.environ.get("TAVILY_API_KEY", "")

openaiclient = OpenAI(api_key=openai_api_key)
tavily_client = TavilyClient(api_key=tavily_api_key)

# Model to use for routing and generation
MODEL = "gpt-4o"

## 1. Internet Tool (Tavily)

In [ ]:
def get_internet_content(user_query: str, action: str) -> str:
    """
    Fetches live search results via Tavily for INTERNET_QUERY routes.

    Args:
        user_query: The user's question.
        action: Expected to be 'INTERNET_QUERY'.

    Returns:
        Formatted string of search results with numbered citations.
    """
    print("Getting your response from the internet 🌐 ...")
    try:
        data = tavily_client.search(query=user_query, max_results=5)
        results = data.get("results", [])

        if not results:
            return "No results found."

        parts = []
        for i, result in enumerate(results, start=1):
            title = result.get("title", "")
            content = result.get("content", "")
            url = result.get("url", "")
            if content:
                parts.append(f"[{i}] {title}\n    {content}\n    Source: {url}")

        return "\n\n".join(parts) if parts else "No results found."

    except Exception as err:
        return f"Search error: {err}"

In [ ]:
# Quick test
# print(get_internet_content("best travel destinations in 2025", "INTERNET_QUERY"))

## 2. Router Query Function

Classifies each query into one of three routes:
- `OPENAI_QUERY` — OpenAI docs, agents, APIs
- `10K_DOCUMENT_QUERY` — Lyft/Uber SEC 10-K filings
- `INTERNET_QUERY` — anything else (live web)

In [ ]:
def route_query(user_query: str) -> dict:
    """
    Classifies user_query into a routing action using the LLM.

    Returns:
        dict with keys: action, reason, answer
    """
    router_system_prompt = f"""
    As a professional query router, classify user input into one of three categories:
    1. "OPENAI_QUERY": Questions about OpenAI agents, models, APIs, guardrails, embeddings.
    2. "10K_DOCUMENT_QUERY": Questions about Lyft or Uber SEC 10-K annual reports.
    3. "INTERNET_QUERY": Everything else requiring real-time or broader web information.

    Always respond in valid JSON:
    {{
        "action": "OPENAI_QUERY" | "10K_DOCUMENT_QUERY" | "INTERNET_QUERY",
        "reason": "brief justification",
        "answer": "AT MAX 5 words. Leave empty if INTERNET_QUERY"
    }}

    User: {user_query}
    """
    try:
        response = openaiclient.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": router_system_prompt}]
        )
        task_response = response.choices[0].message.content
        json_match = re.search(r"\{.*\}", task_response, re.DOTALL)
        return json.loads(json_match.group())

    except (OpenAIError, json.JSONDecodeError, AttributeError) as err:
        return {"action": "INTERNET_QUERY", "reason": str(err), "answer": ""}

In [ ]:
# route_query("what is the revenue of uber in 2021?")

## 3. Qdrant Vector Database

Clone the course repo to get the pre-built collections:
```bash
git clone https://github.com/hamzafarooq/multi-agent-course.git
```
Then set `QDRANT_PATH` to the `Agentic_RAG/qdrant_data` folder inside it.

In [ ]:
QDRANT_PATH = os.environ.get(
    "QDRANT_PATH",
    "./multi-agent-course/modules/Module_3_Production_Agentic_RAG_AI_Systems/Agentic_RAG/qdrant_data"
)

client = qdrant_client.AsyncQdrantClient(path=QDRANT_PATH)

## 4. Embedding Model

In [ ]:
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)


def get_text_embeddings(text: str) -> np.ndarray:
    """
    Converts text to a dense embedding via nomic-embed-text-v1.5 (mean pooling).
    Returns: np.ndarray of shape (hidden_dim,)
    """
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    outputs = text_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].detach().numpy()

## 5. RAG Generator

In [ ]:
def rag_formatted_response(user_query: str, context: list) -> str:
    """
    Generates a grounded answer from retrieved context chunks with numbered citations [1][2].
    """
    rag_prompt = f"""
    Based on the given context, answer the user query: {user_query}
    Context:
    {context}
    Employ references to the ID of articles provided [ID], ensuring their relevance.
    Referencing format: [1][2]...
    """
    response = openaiclient.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": rag_prompt}]
    )
    return response.choices[0].message.content


async def retrieve_and_response(user_query: str, action: str) -> str:
    """
    Embeds the query, retrieves top-3 chunks from Qdrant, generates a RAG response.
    """
    collections = {
        "OPENAI_QUERY": "opnai_data",
        "10K_DOCUMENT_QUERY": "10k_data",
    }
    if action not in collections:
        return "Invalid action type for retrieval."

    try:
        query_vec = get_text_embeddings(user_query)
        hits = await client.query_points(
            collection_name=collections[action],
            query=query_vec,
            limit=3
        )
        contents = [p.payload["content"] for p in hits.points]
        if not contents:
            return "No relevant content found."
        return rag_formatted_response(user_query, contents)
    except Exception as err:
        return f"Retrieval error: {err}"

## 6. Agentic RAG Orchestrator

In [ ]:
routes = {
    "OPENAI_QUERY": retrieve_and_response,
    "10K_DOCUMENT_QUERY": retrieve_and_response,
    "INTERNET_QUERY": get_internet_content,
}


def _execute_route(user_query: str, action: str) -> str:
    """Dispatch helper — handles async vs sync route functions."""
    fn = routes.get(action)
    if not fn:
        return f"Unsupported action: {action}"
    if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
        return asyncio.run(fn(user_query, action))
    return fn(user_query, action)


def agentic_rag(user_query: str) -> None:
    CYAN, GREY, BOLD, RESET = "\033[96m", "\033[90m", "\033[1m", "\033[0m"

    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")
    try:
        response = route_query(user_query)
    except Exception as e:
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\nRouting error: {e}\n")
        return

    action = response.get("action")
    reason = response.get("reason")
    print(f"{GREY}📍 Route: {action}\n📝 Reason: {reason}\n⚙️  Processing...{RESET}\n")

    result = _execute_route(user_query, action)
    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n{result}\n")

In [ ]:
# agentic_rag("what was uber revenue in 2021?")
# agentic_rag("how to work with chat completions?")
# agentic_rag("List me new LLMs in 2025")

## 7. Role-Based Access Control (RBAC)

| User  | Role             | OPENAI_QUERY | 10K_DOCUMENT_QUERY | INTERNET_QUERY |
|-------|------------------|:---:|:---:|:---:|
| alice | engineer         | ✅  | ❌  | ✅  |
| bob   | finance_analyst  | ✅  | ✅  | ❌  |

In [ ]:
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}

ROLE_PERMISSIONS = {
    "engineer":        {"OPENAI_QUERY", "INTERNET_QUERY"},
    "finance_analyst": {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"},
}

SOURCE_LABELS = {
    "OPENAI_QUERY":       "OpenAI documentation",
    "10K_DOCUMENT_QUERY": "10-K financial filings",
    "INTERNET_QUERY":     "live internet search",
}


def has_access(user_id: str, action: str) -> bool:
    role = USERS.get(user_id)
    return role is not None and action in ROLE_PERMISSIONS.get(role, set())


def allowed_sources(user_id: str) -> set:
    return ROLE_PERMISSIONS.get(USERS.get(user_id), set())

In [ ]:
def secure_agentic_rag(user_id: str, user_query: str) -> str:
    """
    RBAC-gated orchestrator.
    Order: identity check → route → RBAC gate → retrieve/respond.
    """
    CYAN, GREY, RED, GREEN, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"
    )
    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    if role is None:
        msg = f"🚫 Access denied: unknown user '{user_id}'."
        print(f"{RED}{msg}{RESET}\n")
        return msg

    decision = route_query(user_query)
    action = decision.get("action")
    print(f"{GREY}📍 Route: {action}\n📝 Reason: {decision.get('reason')}{RESET}\n")

    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        msg = f"🚫 Access denied: role '{role}' cannot query {source}."
        print(f"{RED}{msg}{RESET}\n")
        return msg

    print(f"{GREEN}✅ Access granted{RESET} — processing...\n")
    result = _execute_route(user_query, action)
    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n{result}\n")
    return result

In [ ]:
# secure_agentic_rag("alice", "what was lyft revenue in 2022?")  # DENIED
# secure_agentic_rag("bob",   "what was lyft revenue in 2022?")  # ALLOWED

---
## Part 1 — Sub-Query Division

Compound questions are split into independent sub-questions. Each sub-question is routed and answered separately, then the results are combined into one coherent answer with globally renumbered citations.

| Input | Sub-queries | Routes |
|---|---|---|
| `"what was uber revenue in 2021?"` | 1 | `10K_DOCUMENT_QUERY` |
| `"what was lyft revenue in 2021 and what was uber revenue in 2021"` | 2 | both `10K_DOCUMENT_QUERY` |
| `"what was uber's 2021 revenue and what are the newest LLMs?"` | 2 | `10K_DOCUMENT_QUERY` + `INTERNET_QUERY` |

In [ ]:
def sub_queries(user_query: str) -> str:
    """
    Reference implementation (from original notebook).
    Returns the raw LLM response string — a JSON object with a 'subQuestions' list.
    """
    prompt = f"""
    You are a query router. If the input contains multiple distinct questions, break it into
    sub-questions. Otherwise, keep it as one. Return a JSON object like:

    {{"subQuestions": ["..."]}}

    Query: "{user_query}"
    Output:
    """
    response = openaiclient.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
def _parse_sub_queries(raw: str, fallback: str) -> list[str]:
    """
    Defensively parse the LLM JSON response from sub_queries().
    Falls back to [fallback] (the original query) on any parse failure.
    """
    try:
        json_match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not json_match:
            return [fallback]
        parsed = json.loads(json_match.group())
        questions = parsed.get("subQuestions", [])
        return questions if questions else [fallback]
    except (json.JSONDecodeError, AttributeError):
        return [fallback]


def _synthesize_answers(original_query: str, sub_results: list[dict]) -> str:
    """
    Merge answers from multiple sub-queries into one coherent response.
    Renumbers citations globally across all sub-answers.
    """
    if len(sub_results) == 1:
        return sub_results[0]["answer"]

    context_parts = []
    for i, r in enumerate(sub_results, 1):
        context_parts.append(
            f"Sub-question {i}: {r['query']}\n"
            f"Route: {r['action']}\n"
            f"Answer: {r['answer']}"
        )

    synthesis_prompt = f"""
    You are given multiple sub-questions and their individual answers.
    Combine them into a single coherent response that fully addresses the original question.
    Preserve all factual content. Renumber citations globally as [1][2][3]... in the order
    they first appear across all sub-answers.

    Original question: {original_query}

    {chr(10).join(context_parts)}

    Provide a unified answer:
    """
    response = openaiclient.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": synthesis_prompt}]
    )
    return response.choices[0].message.content


def agentic_rag_multi(user_query: str) -> str:
    """
    Split a compound query, route and answer each sub-query independently,
    then synthesise one final answer preserving all citations.

    Args:
        user_query: Possibly compound question.

    Returns:
        Single composed answer covering every sub-question.
    """
    CYAN, GREY, BOLD, RESET = "\033[96m", "\033[90m", "\033[1m", "\033[0m"

    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

    # Step 1: split into sub-queries (defensively parsed)
    raw = sub_queries(user_query)
    queries = _parse_sub_queries(raw, user_query)
    print(f"{GREY}🔀 Sub-queries ({len(queries)}): {queries}{RESET}\n")

    # Step 2: route and answer each sub-query
    sub_results = []
    for i, sq in enumerate(queries, 1):
        print(f"{GREY}  [{i}/{len(queries)}] Routing: '{sq}'{RESET}")
        try:
            decision = route_query(sq)
            action = decision.get("action", "INTERNET_QUERY")
            print(f"{GREY}         → {action}{RESET}")
            answer = _execute_route(sq, action)
        except Exception as e:
            action = "ERROR"
            answer = f"Error processing sub-query: {e}"

        sub_results.append({"query": sq, "action": action, "answer": answer})

    print()

    # Step 3: compose final answer
    final = _synthesize_answers(user_query, sub_results)
    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n{final}\n")
    return final

In [ ]:
# Test 1: single question — should behave like agentic_rag
# agentic_rag_multi("what was uber revenue in 2021?")

In [ ]:
# Test 2: two questions, same route
# agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")

In [ ]:
# Test 3: two questions, different routes
# agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")

---
## Bonus — RBAC-Aware Semantic Cache

**Design choice: Partitioned cache (one FAISS index per role)**

Each role gets its own FAISS `IndexFlatL2` over normalized embeddings. A lookup only searches within the requesting user's role partition — cross-role cache hits are physically impossible because the indexes are separate. This eliminates the need for per-entry permission checks at lookup time, and ensures that even semantically similar queries by users with different roles never share cached answers.

Trade-off: users with overlapping permissions (e.g., both alice and bob can use `OPENAI_QUERY`) don't share their cached entries. This wastes some compute but maximises isolation — acceptable for a security-critical system.

**Cache suppression rules:**
- `INTERNET_QUERY` results are never cached (live data).
- `DENIED` responses are never cached.
- Cache lookup is always role-partitioned — a near-paraphrase from alice never hits bob's rows.

In [ ]:
class RoleAwareSemanticCache:
    """
    Semantic cache partitioned by role.
    One FAISS IndexFlatL2 per role; lookups are confined to the requesting user's partition.

    Embeddings are L2-normalised before storage so that Euclidean distance tracks
    cosine similarity (distance 0 = identical, distance √2 ≈ 1.41 = orthogonal).
    The default threshold of 0.2 corresponds to cosine similarity ≥ 0.98.
    Raise it (e.g. 0.5) for more aggressive caching of near-paraphrases.
    """

    def __init__(self, threshold: float = 0.2):
        self.threshold = threshold
        # role -> {"index": faiss.Index, "answers": list[str], "questions": list[str]}
        self._partitions: dict[str, dict] = {}

    # ── internal ──────────────────────────────────────────────────────────────

    @staticmethod
    def _normalize(vec: np.ndarray) -> np.ndarray:
        """L2-normalise a 1-D numpy array and return it as float32."""
        vec = vec.astype(np.float32)
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def _partition(self, role: str, dim: int) -> dict:
        """Return (and lazily create) the FAISS partition for this role."""
        if role not in self._partitions:
            self._partitions[role] = {
                "index": faiss.IndexFlatL2(dim),
                "answers": [],
                "questions": [],
            }
        return self._partitions[role]

    # ── public API ────────────────────────────────────────────────────────────

    def check(self, user_id: str, question: str):
        """
        Look up question in the user's role partition.

        Returns:
            (hit: bool, answer: str | None, embedding: np.ndarray, distance: float | None)
            The embedding is always returned so the caller can reuse it in add().
        """
        role = USERS.get(user_id)
        embedding = self._normalize(get_text_embeddings(question))

        if role is None or role not in self._partitions:
            return False, None, embedding, None

        partition = self._partitions[role]
        if partition["index"].ntotal == 0:
            return False, None, embedding, None

        query_vec = embedding.reshape(1, -1)
        distances, indices = partition["index"].search(query_vec, 1)
        distance = float(distances[0][0])
        idx = int(indices[0][0])

        if distance <= self.threshold:
            return True, partition["answers"][idx], embedding, distance

        return False, None, embedding, distance

    def add(self, user_id: str, question: str, answer: str, embedding: np.ndarray | None = None):
        """
        Store an answer in the user's role partition.
        Accepts a pre-computed embedding from check() to avoid a second model call.
        """
        role = USERS.get(user_id)
        if role is None:
            return

        if embedding is None:
            embedding = self._normalize(get_text_embeddings(question))

        vec = embedding.reshape(1, -1).astype(np.float32)
        dim = vec.shape[1]
        partition = self._partition(role, dim)
        partition["index"].add(vec)
        partition["answers"].append(answer)
        partition["questions"].append(question)

In [ ]:
def secure_agentic_rag_cached(user_id: str, user_query: str, cache: RoleAwareSemanticCache) -> dict:
    """
    RBAC-gated agentic RAG with role-partitioned semantic cache.

    Order of operations:
        1. Identity check    — unknown users denied immediately
        2. Route query       — LLM decides the knowledge source
        3. RBAC gate         — deny if role lacks permission for that source
        4. Cache lookup      — skip for INTERNET_QUERY (live data must not be cached)
        5. Pipeline          — retrieve + generate on cache miss
        6. Cache store       — persist answer in the user's role partition

    Returns:
        dict: {"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}
    """
    CYAN, RED, GREEN, BOLD, RESET = "\033[96m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"

    # Step 1 — identity
    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤{RESET} {user_id} (role: {role or 'UNKNOWN'}) | {user_query}")

    if role is None:
        msg = f"🚫 Access denied: unknown user '{user_id}'."
        print(f"{RED}{msg}{RESET}")
        return {"answer": msg, "status": "DENIED", "role": None}

    # Step 2 — route
    try:
        decision = route_query(user_query)
        action = decision.get("action", "INTERNET_QUERY")
    except Exception as e:
        msg = f"Routing error: {e}"
        return {"answer": msg, "status": "DENIED", "role": role}

    # Step 3 — RBAC gate
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        msg = f"🚫 Access denied: role '{role}' cannot query {source}."
        print(f"{RED}{msg}{RESET}")
        return {"answer": msg, "status": "DENIED", "role": role}

    # Step 4 — cache lookup (skip for live internet data)
    if action != "INTERNET_QUERY":
        hit, cached_answer, embedding, distance = cache.check(user_id, user_query)
        if hit:
            print(f"{GREEN}⚡ Cache HIT (distance={distance:.4f}){RESET}")
            return {"answer": cached_answer, "status": "HIT", "role": role}
        print(f"  Cache MISS (distance={distance})")
    else:
        embedding = None

    # Step 5 — execute pipeline
    try:
        result = _execute_route(user_query, action)
    except Exception as e:
        return {"answer": f"Execution error: {e}", "status": "MISS", "role": role}

    # Step 6 — store in cache (not for internet queries)
    if action != "INTERNET_QUERY":
        cache.add(user_id, user_query, result, embedding)

    print(f"{CYAN}✅ MISS → stored in cache{RESET}")
    return {"answer": result, "status": "MISS", "role": role}

## Self-Check

In [ ]:
def run_self_check():
    cache = RoleAwareSemanticCache()
    q_fin = "what was uber revenue in 2021?"
    q_doc = "how do I build an agent with the OpenAI Agents SDK?"

    # 1. bob may read financials — first ask is a MISS
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "MISS", f"expected MISS, got {r['status']}"

    # 2. bob asks again — served from cache
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "HIT", f"expected HIT, got {r['status']}"

    # 3. THE LEAK TEST — alice must be denied, never served bob's cached answer
    r = secure_agentic_rag_cached("alice", q_fin, cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} on finance data"

    # 4. near-paraphrase must also be denied, not matched into bob's rows
    r = secure_agentic_rag_cached("alice", "how much revenue did Uber make in 2021?", cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} via paraphrase"

    # 5. unknown users are rejected outright
    r = secure_agentic_rag_cached("carol", q_doc, cache)
    assert r["status"] == "DENIED", f"expected DENIED for unknown user, got {r['status']}"

    # 6. shared source still caches normally within a role
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "HIT"

    print("\n✅ All checks passed — cache is fast and does not leak across roles.")


# run_self_check()